# Module 6 — Weak Supervision with Snorkel (Windows Logs, Full Lab)

**Pipeline:** 1) Load larger dataset (online or manual) → 2) LFs → LabelModel → 3) Classifier + P/R → 4) Export Sigma

## 0) Environment
```bash
pip install snorkel pandas numpy scikit-learn matplotlib
```

## 1) Load a Larger Dataset
Try downloading Windows logs from public datasets or set LOCAL_PATH manually.

In [21]:
LOCAL_PATH = "./ProcessCreation.csv"  # e.g., r"C:\\Users\\you\\Downloads\\windows_events.csv"
if 'df_raw' not in globals() or df_raw is None:
    import os
    if LOCAL_PATH and os.path.exists(LOCAL_PATH):
        df_raw = pd.read_csv(LOCAL_PATH); used_url = "local_file"
        print("[+] Loaded local dataset:", LOCAL_PATH, "shape=", df_raw.shape)
        display(df_raw.head(3))
    else:
        print("Set LOCAL_PATH to a valid CSV path.")

## 2) Prepare Data

In [22]:
# 2) Prepare Data — robust to your CSV schema and 4688 message format

import re
import numpy as np
import pandas as pd

df = df_raw.copy()

# If the CSV used the event level (e.g., "Audit Success") as the index, bring it back as a column.
if not isinstance(df.index, pd.RangeIndex):
    df = df.reset_index().rename(columns={"index": "Level"})
elif df.index.name:
    df = df.reset_index().rename(columns={df.index.name: "Level"})

# Normalize column names for easier matching (keep original too)
orig_cols = {c: c for c in df.columns}
norm_map = {c: c.strip().lower().replace("_", " ").replace("-", " ") for c in df.columns}
inv_map = {}
for k, v in norm_map.items():
    # If duplicates after normalization, keep the first
    inv_map.setdefault(v, k)

def has(colname):
    return colname in inv_map

def col(colname, default=None):
    return inv_map.get(colname, default)

# Common Windows CSV headers
# - "Message" may be missing; your preview shows the long text in "Task Category"
text_col = None
for candidate in ["message", "task category", "details", "description"]:
    if has(candidate):
        text_col = col(candidate)
        break

# Optional helpful fields
provider_col = col("source") or col("provider")
event_id_col = col("event id") or col("eventid") or col("id")
timestamp_col = col("date and time") or col("time created") or col("timestamp")
level_col = "Level" if "Level" in df.columns else (col("level") or col("keywords"))

# Ensure text column exists
if not text_col:
    # Last resort: concatenate any object-typed columns
    obj_cols = [c for c in df.columns if df[c].dtype == object]
    df["message"] = df[obj_cols].astype(str).agg(" ".join, axis=1).str.strip()
    text_col = "message"
else:
    # Standardize to 'message'
    df["message"] = df[text_col].astype(str)

# Fill missing helper columns (so downstream LFs have something to read)
for c in ["process_name", "parent_process_name", "command_line", "channel", "provider", "computer", "user", "text"]:
    if c not in df.columns:
        df[c] = ""

if provider_col:
    df["provider"] = df[provider_col].astype(str)
if event_id_col:
    df["event_id"] = df[event_id_col].astype(str)
if timestamp_col:
    df["timestamp"] = pd.to_datetime(df[timestamp_col], errors="coerce").astype("str")
if level_col and level_col in df.columns:
    df["level"] = df[level_col].astype(str)

# Parse common fields from Windows Security 4688 / Sysmon Event 1 message bodies
# Examples in messages:
#   New Process Name: C:\Windows\System32\cmd.exe
#   Creator Process Name: C:\Windows\explorer.exe
#   Command Line: "C:\Windows\System32\cmd.exe" /c whoami
# Or Sysmon:
#   Image: C:\Windows\System32\cmd.exe
#   ParentImage: C:\Windows\explorer.exe
#   CommandLine: ...
proc_patterns = [
    r"(?i)\bNew Process Name\s*:\s*(.+)",
    r"(?i)\bImage\s*:\s*(.+)",
]
parent_patterns = [
    r"(?i)\bCreator Process Name\s*:\s*(.+)",
    r"(?i)\bParent Image\s*:\s*(.+)",
    r"(?i)\bParentImage\s*:\s*(.+)",
]
cmd_patterns = [
    r"(?i)\bCommand Line\s*:\s*(.+)",
    r"(?i)\bCommandLine\s*:\s*(.+)",
]

def first_match(text, patterns):
    for p in patterns:
        m = re.search(p, text, flags=re.DOTALL)
        if m:
            return m.group(1).strip()
    return ""

def extract_fields(msg):
    m = str(msg or "")
    return pd.Series(
        {
            "process_name": first_match(m, proc_patterns),
            "parent_process_name": first_match(m, parent_patterns),
            "command_line": first_match(m, cmd_patterns),
            "text": m.strip(),
        }
    )

parsed = df["message"].apply(extract_fields)
# Only fill if empty to avoid overwriting when columns already exist
for k in ["process_name", "parent_process_name", "command_line", "text"]:
    empty_mask = (df[k].astype(str).str.len() == 0)
    df.loc[empty_mask, k] = parsed.loc[empty_mask, k]

# Final clean-up
df.replace({np.nan: ""}, inplace=True)
df["message"] = df["message"].astype(str).str.strip()
df = df[df["message"].str.len() > 0].reset_index(drop=True)

print("Rows:", len(df))
display(df[["message", "process_name", "parent_process_name", "command_line"]].head(3))

Rows: 219


,message,process_name,parent_process_name,command_line
0,A new process has been created.\r\n\r\nCreator...,C:\Windows\System32\smss.exe\r\n\tToken Elevat...,C:\Windows\System32\smss.exe\r\n\tProcess Comm...,Token Elevation Type indicates the type of tok...
1,A new process has been created.\r\n\r\nCreator...,C:\Windows\System32\csrss.exe\r\n\tToken Eleva...,C:\Windows\System32\smss.exe\r\n\tProcess Comm...,Token Elevation Type indicates the type of tok...
2,A new process has been created.\r\n\r\nCreator...,C:\Windows\System32\wininit.exe\r\n\tToken Ele...,C:\Windows\System32\smss.exe\r\n\tProcess Comm...,Token Elevation Type indicates the type of tok...


## 3) Labeling Functions (LFs)

In [23]:
# Extra LFs for 4688 "Details" patterns, then append to existing lfs and re-apply
import re
from snorkel.labeling import labeling_function, PandasLFApplier, LFAnalysis

ABSTAIN, SUSP, BENIGN = -1, 1, 0

def txt(x, name):
    try:
        return (getattr(x, name) or "").lower()
    except Exception:
        return ""

@labeling_function(name="lf_ps_execpolicy_bypass")
def lf_ps_execpolicy_bypass(x):
    cl = txt(x, "command_line"); pn = txt(x, "process_name"); m = txt(x, "text") or txt(x, "message")
    hit_cl = ("powershell" in pn or "powershell" in cl) and (
        "set-executionpolicy" in cl or "-executionpolicy" in cl or "scope process bypass" in cl or " bypass " in f" {cl} "
    )
    hit_m = ("powershell" in m and "set-executionpolicy" in m and "bypass" in m)
    return SUSP if (hit_cl or hit_m) else ABSTAIN

@labeling_function(name="lf_ps1_from_downloads")
def lf_ps1_from_downloads(x):
    cl = txt(x, "command_line"); m = txt(x, "text") or txt(x, "message")
    in_dl = (".ps1" in cl and r"\\downloads\\" in cl) or (".ps1" in m and r"\\downloads\\" in m)
    amp = re.search(r"&\s*['\"]?[a-z]:\\\\users\\\\[^\\]+\\\\downloads\\\\[^'\"]+\.ps1", cl) is not None
    return SUSP if (in_dl or amp) else ABSTAIN

@labeling_function(name="lf_explorer_spawns_powershell")
def lf_explorer_spawns_powershell(x):
    parent = txt(x, "parent_process_name"); pn = txt(x, "process_name")
    return SUSP if ("explorer.exe" in parent and "powershell.exe" in pn) else ABSTAIN

@labeling_function(name="lf_high_integrity_powershell")
def lf_high_integrity_powershell(x):
    m = txt(x, "text") or txt(x, "message"); pn = txt(x, "process_name")
    return SUSP if ("powershell.exe" in pn and "mandatory label\\high mandatory level" in m) else ABSTAIN

@labeling_function(name="lf_benign_core_services")
def lf_benign_core_services(x):
    pn = txt(x, "process_name")
    return BENIGN if ("\\svchost.exe" in pn or "\\conhost.exe" in pn) else ABSTAIN

lfs_extra = [
    lf_ps_execpolicy_bypass,
    lf_ps1_from_downloads,
    lf_explorer_spawns_powershell,
    lf_high_integrity_powershell,
    lf_benign_core_services,
]

# Append to existing LFs if present
try:
    lfs = list(lfs) + lfs_extra
except NameError:
    lfs = lfs_extra

applier = PandasLFApplier(lfs)
L = applier.apply(df)
LFAnalysis(L=L, lfs=lfs).lf_summary()

100%|██████████| 219/219 [00:00<00:00, 4315.74it/s]


,j,Polarity,Coverage,Overlaps,Conflicts
lf_powershell_encoded,0,[],0.000000,0.000000,0.000000
lf_certutil_download,1,[],0.000000,0.000000,0.000000
lf_mshta_url,2,[],0.000000,0.000000,0.000000
lf_bitsadmin,3,[],0.000000,0.000000,0.000000
lf_word_spawns_ps,4,[],0.000000,0.000000,0.000000
lf_update_benign,5,[],0.000000,0.000000,0.000000
lf_ps_execpolicy_bypass,6,[1],0.009132,0.009132,0.000000
lf_ps1_from_downloads,7,[],0.000000,0.000000,0.000000
lf_explorer_spawns_powershell,8,[1],0.018265,0.018265,0.000000
lf_high_integrity_powershell,9,[1],0.054795,0.036530,0.018265


## 4) LabelModel

In [25]:
from snorkel.labeling.model import LabelModel
import numpy as np

label_model = LabelModel(cardinality=2, verbose=True)
label_model.fit(L_train=L, n_epochs=300, log_freq=100, seed=7)

# Keep probabilities + initial weak label
p_susp = label_model.predict_proba(L)[:, 1]
df["p_susp"] = p_susp
df["weak_label"] = (p_susp >= 0.5).astype(int)

df[["process_name","parent_process_name","command_line","p_susp","weak_label"]].head()

INFO:root:Computing O...
INFO:root:Estimating \mu...
100%|██████████| 300/300 [00:00<00:00, 748.63epoch/s]
INFO:root:Finished Training


,process_name,parent_process_name,command_line,p_susp,weak_label
0,C:\Windows\System32\smss.exe\r\n\tToken Elevat...,C:\Windows\System32\smss.exe\r\n\tProcess Comm...,Token Elevation Type indicates the type of tok...,0.5,1
1,C:\Windows\System32\csrss.exe\r\n\tToken Eleva...,C:\Windows\System32\smss.exe\r\n\tProcess Comm...,Token Elevation Type indicates the type of tok...,0.5,1
2,C:\Windows\System32\wininit.exe\r\n\tToken Ele...,C:\Windows\System32\smss.exe\r\n\tProcess Comm...,Token Elevation Type indicates the type of tok...,0.5,1
3,C:\Windows\System32\autochk.exe\r\n\tToken Ele...,C:\Windows\System32\smss.exe\r\n\tProcess Comm...,Token Elevation Type indicates the type of tok...,0.5,1
4,C:\Windows\System32\smss.exe\r\n\tToken Elevat...,C:\Windows\System32\smss.exe\r\n\tProcess Comm...,Token Elevation Type indicates the type of tok...,0.5,1


## 5) Train Classifier + Evaluation

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np

assert "p_susp" in df.columns, "Run the LabelModel cell first so df['p_susp'] exists."

# Keep only confident examples to form both classes
t_pos, t_neg = 0.7, 0.3
pos_idx = df["p_susp"] >= t_pos
neg_idx = df["p_susp"] <= t_neg

train = df[pos_idx | neg_idx].copy()
train["weak_label"] = (train["p_susp"] >= t_pos).astype(int)

# Fallback: use top/bottom quantiles if one class remains
if train["weak_label"].nunique() < 2:
    hi_q, lo_q = df["p_susp"].quantile(0.9), df["p_susp"].quantile(0.1)
    train = df[(df["p_susp"] >= hi_q) | (df["p_susp"] <= lo_q)].copy()
    train["weak_label"] = (train["p_susp"] >= hi_q).astype(int)

print("Training class distribution:", train["weak_label"].value_counts().to_dict())
assert train["weak_label"].nunique() == 2, "Still one class; loosen thresholds (e.g., 0.7/0.3) or add more LFs."

# Vectorize + split
vec = TfidfVectorizer(ngram_range=(1,2), min_df=2)
X = vec.fit_transform(train["message"])
y = train["weak_label"].values

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=7, stratify=y)
clf = LogisticRegression(max_iter=400, class_weight="balanced").fit(Xtr, ytr)
pred = clf.predict(Xte)

print("Holdout size:", len(yte))
print(classification_report(yte, pred, digits=3))

Training class distribution: {1: 188, 0: 27}
Holdout size: 65
              precision    recall  f1-score   support

           0      0.800     1.000     0.889         8
           1      1.000     0.965     0.982        57

    accuracy                          0.969        65
   macro avg      0.900     0.982     0.936        65
weighted avg      0.975     0.969     0.971        65



## 6) Export Sigma Rules

In [ ]:
import textwrap

rules = []

def add_rule(title, selection, level="high"):
    rules.append(
        textwrap.dedent(f"""
        ---
        title: {title}
        status: experimental
        logsource:
          product: windows
          category: process_creation
        detection:
          selection:
{selection}
          condition: selection
        level: {level}
        """).lstrip()
    )

# Seeds you had
add_rule(
    "Suspicious Certutil Download (WeakSup Seed)",
    "        Image|endswith: '\\\\cmd.exe'\\n"
    "        CommandLine|contains|all:\\n"
    "          - 'certutil'\\n"
    "          - 'http'"
)
add_rule(
    "MSHTA with Remote URL (WeakSup Seed)",
    "        Image|endswith: '\\\\mshta.exe'\\n"
    "        CommandLine|contains: 'http'"
)
add_rule(
    "PowerShell EncodedCommand (WeakSup Seed)",
    "        Image|endswith: '\\\\powershell.exe'\\n"
    "        CommandLine|contains: '-EncodedCommand'"
)
add_rule(
    "WinWord Spawns PowerShell (WeakSup Seed)",
    "        ParentImage|endswith: '\\\\WINWORD.EXE'\\n"
    "        Image|endswith: '\\\\powershell.exe'"
)
add_rule(
    "BITSAdmin File Transfer (WeakSup Seed)",
    "        CommandLine|contains: 'bitsadmin'"
)

# New rules aligned with LFs and payload.ps1 behavior for the demonstration. trans
add_rule(
    "PowerShell ExecutionPolicy Bypass (WeakSup Seed)",
    "        Image|endswith: '\\\\powershell.exe'\\n"
    "        CommandLine|contains|all:\\n"
    "          - 'executionpolicy'\\n"
    "          - 'bypass'"
)
add_rule(
    "PowerShell Executes PS1 from Downloads (WeakSup Seed)",
    "        Image|endswith: '\\\\powershell.exe'\\n"
    "        CommandLine|contains|all:\\n"
    "          - '\\\\Users\\\\'\\n"
    "          - '\\\\Downloads\\\\'\\n"
    "          - '.ps1'"
)
add_rule(
    "Explorer Spawns PowerShell (WeakSup Seed)",
    "        ParentImage|endswith: '\\\\explorer.exe'\\n"
    "        Image|endswith: '\\\\powershell.exe'"
)

sigma_path = "sigma_from_weak_supervision.yml"
with open(sigma_path, "w", encoding="utf-8") as f:
    f.write("\n".join(rules))
print("Saved Sigma draft(s) to:", sigma_path)

Saved Sigma draft(s) to: sigma_from_weak_supervision.yml
